In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
EXCEL_PATH = Path("CHE447_27_case_auto_calc_template.xlsx")
SHEET_NAME = "Case_Auto"

FEATURE_COLS = [
    "Tinf_K",
    "h_conv_W_m2K",
    "rho_ice_kg_m3",
    "Cp_ice_J_kgK",
    "dH_melt_J_kg",
    "t_cone_mm",
    "Deff_eff_m2_s",
]

TARGET_COLS = [
    "t_melt_onset_s",
    "t_noticeable_soggy_s",
    "t_unacceptably_soggy_s",
    "t_finish_s",
]

In [ ]:
df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

needed_cols = FEATURE_COLS + TARGET_COLS
df_model = df.dropna(subset=needed_cols).copy()

X = df_model[FEATURE_COLS]
y = df_model[TARGET_COLS]

NameError: name 'pd' is not defined

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestRegressor(
        n_estimators=300,
        max_depth=4,
        random_state=42
    ))
])

loo = LeaveOneOut()
y_pred = cross_val_predict(model, X, y, cv=loo)

pred_df = pd.DataFrame(y_pred, columns=[f"pred_{c}" for c in TARGET_COLS], index=df_model.index)
pred_df.head()

In [ ]:
rows = []
for i, target in enumerate(TARGET_COLS):
    actual = y.iloc[:, i].values
    pred = y_pred[:, i]

    rows.append({
        "target": target,
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": mean_squared_error(actual, pred, squared=False),
        "R2": r2_score(actual, pred),
    })

metrics_df = pd.DataFrame(rows)
metrics_df

NameError: name 'TARGET_COLS' is not defined

In [ ]:
for i, target in enumerate(TARGET_COLS):
    actual = y.iloc[:, i].values
    pred = y_pred[:, i]

    plt.figure(figsize=(5, 5))
    plt.scatter(actual, pred)
    lo = min(actual.min(), pred.min())
    hi = max(actual.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], linestyle="--")
    plt.xlabel(f"Actual {target}")
    plt.ylabel(f"Predicted {target}")
    plt.title(target)
    plt.tight_layout()
    plt.show()

In [ ]:
model.fit(X, y)

rf = model.named_steps["rf"]
importance_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": rf.feature_importances_,
}).sort_values("importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(importance_df["feature"], importance_df["importance"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Importance")
plt.title("Feature importance")
plt.tight_layout()
plt.show()